In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
exchange_list = ['binance', 'gate']
coin_list = ['BTC', 'ETH', 'SOL']

In [3]:
log_return_color = "#1f77b4"   # blue
volume_color = "#ff7f0e"       # orange

rows = len(exchange_list)
cols = len(coin_list)

fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=False,
    subplot_titles=[
        f"{exchange.capitalize()} {coin}USDT"
        for exchange in exchange_list
        for coin in coin_list
    ],
    specs=[
        [{"secondary_y": True} for _ in range(cols)]
        for _ in range(rows)
    ]
)

for i, exchange in enumerate(exchange_list, start=1):
    for j, coin in enumerate(coin_list, start=1):

        # Load
        df = pd.read_pickle(f"./raw_data_{exchange}_{coin}USDT.pkl")
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        # 1H buckets (use floor)
        df['ts_1h'] = df['timestamp'].dt.floor('1h')

        df_grouped = df.groupby('ts_1h').agg({
            'price': 'last',
            'quantity_usdt': 'sum'
        })

        # Log returns
        df_grouped['log_return'] = np.log(
            df_grouped['price'] / df_grouped['price'].shift(1)
        )

        # Log returns (fixed color)
        fig.add_trace(
            go.Scatter(
                x=df_grouped.index,
                y=df_grouped['log_return'],
                name=f'{exchange}-{coin} Log Returns',
                mode='lines',
                opacity=0.9,
                line=dict(color=log_return_color),
                showlegend=False
            ),
            row=i,
            col=j,
            secondary_y=False
        )

        # Volume (fixed color)
        fig.add_trace(
            go.Bar(
                x=df_grouped.index,
                y=df_grouped['quantity_usdt'],
                name=f'{exchange}-{coin} Volume USD',
                opacity=0.5,
                marker_color=volume_color,  # FIXED COLOR
                showlegend=False
            ),
            row=i,
            col=j,
            secondary_y=True
        )

# Layout
fig.update_layout(
    title="1H Log Returns (left) vs Volume USD (right)",
    template="plotly_dark",
    bargap=0,
    height=800,
    width=1400
)

# Label axes
for i in range(1, rows + 1):
    fig.update_yaxes(title_text="Log Returns", row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Volume USD", row=i, col=cols, secondary_y=True)

for j in range(1, cols + 1):
    fig.update_xaxes(title_text="Time", row=rows, col=j)

fig.show(renderer="browser")

In [4]:
returns_dfs = []
volume_dfs = []

for exchange in exchange_list:
    for coin in coin_list:
        
        # Load raw data
        df = pd.read_pickle(f"./raw_data_{exchange}_{coin}USDT.pkl")
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Aggregate to 1H (or your desired timeframe)
        df['ts_1h'] = df['timestamp'].dt.floor('1h')
        df_g = df.groupby('ts_1h').agg({
            'price': 'last',
            'quantity_usdt': 'sum'
        })
        
        # Compute log returns
        df_g['log_return'] = np.log(df_g['price'] / df_g['price'].shift(1))
        
        # Rename columns for merging
        returns_dfs.append(
            df_g[['log_return']]
            .rename(columns={'log_return': f'ret_{exchange}_{coin}'})
        )
        
        volume_dfs.append(
            df_g[['quantity_usdt']]
            .rename(columns={'quantity_usdt': f'vol_{exchange}_{coin}'})
        )
        
df_returns = pd.concat(returns_dfs, axis=1)
df_volumes = pd.concat(volume_dfs, axis=1)
corr_returns = df_returns.corr()
corr_volumes = df_volumes.corr()

In [5]:
import plotly.express as px

# --- Returns ---
fig_returns = px.imshow(
    corr_returns,
    text_auto=True,
    aspect="auto",
    title="Correlation Matrix of Log Returns"
)
fig_returns.show(renderer="browser")

# --- Volumes ---
fig_vol = px.imshow(
    corr_volumes,
    text_auto=True,
    aspect="auto",
    title="Correlation Matrix of Volumes (USD)"
)
fig_vol.show(renderer="browser")

In [7]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch
from arch.unitroot import ADF
from arch import arch_model
import numpy as np
import warnings

def stationarity_tests(series, name="series"):
    print(f"\n===== {name} =====")

    # Drop nan
    x = series.dropna()

    # ADF (null: non-stationary / unit root)
    adf_res = adfuller(x)
    print(f"\nADF Test (H0: unit root, non-stationary)")
    print(f"ADF Statistic: {adf_res[0]:.4f}")
    print(f"p-value      : {adf_res[1]:.4f}")

    print("\nKPSS Test (H0: stationary)")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # suppress interpolation warnings
            kpss_stat, kpss_p, lags, crit = kpss(x, regression="c", nlags="auto")

        print(f"KPSS Statistic: {kpss_stat:.4f}")
        print(f"p-value       : {kpss_p:.4f}")
        print(f"Lags used     : {lags}")

    except ValueError as e:
        print("KPSS test failed:", e)

    print("\nARCH LM Test (H0: no ARCH effects / constant variance)")
    try:
        # rule-of-thumb: 12 lags for hourly (half-day), or use something like 24 (1 day)
        arch_lags = min(24, max(1, len(x)//50))  # safe default that scales with sample size
        lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(x, nlags=arch_lags)

        print(f"LM Statistic  : {lm_stat:.4f}")
        print(f"LM p-value    : {lm_pvalue:.4f}")
        print(f"F Statistic   : {f_stat:.4f}")
        print(f"F p-value     : {f_pvalue:.4f}")
        print(f"Lags used     : {arch_lags}")
    except Exception as e:
        print("ARCH test failed:", e)

In [8]:
for col in df_returns.columns:
    stationarity_tests(df_returns[col], name=col)


===== ret_binance_BTC =====

ADF Test (H0: unit root, non-stationary)
ADF Statistic: -9.2466
p-value      : 0.0000

KPSS Test (H0: stationary)
KPSS Statistic: 0.5236
p-value       : 0.0363
Lags used     : 7

ARCH LM Test (H0: no ARCH effects / constant variance)
LM Statistic  : 6.3278
LM p-value    : 0.0423
F Statistic   : 3.2421
F p-value     : 0.0421
Lags used     : 2

===== ret_binance_ETH =====

ADF Test (H0: unit root, non-stationary)
ADF Statistic: -8.5947
p-value      : 0.0000

KPSS Test (H0: stationary)
KPSS Statistic: 0.2514
p-value       : 0.1000
Lags used     : 13

ARCH LM Test (H0: no ARCH effects / constant variance)
LM Statistic  : 25.7403
LM p-value    : 0.0000
F Statistic   : 15.4094
F p-value     : 0.0000
Lags used     : 2

===== ret_binance_SOL =====

ADF Test (H0: unit root, non-stationary)
ADF Statistic: -10.3242
p-value      : 0.0000

KPSS Test (H0: stationary)
KPSS Statistic: 0.2322
p-value       : 0.1000
Lags used     : 7

ARCH LM Test (H0: no ARCH effects / con

In [9]:
exchange = exchange_list[0]
coin = coin_list[0]

In [10]:
df = pd.read_pickle(f"./raw_data_{exchange}_{coin}USDT.pkl")

In [11]:
df.head(5)

,timestamp,price,quantity,taker_side,quantity_usdt
0,2025-09-25 00:00:00.010,113307.01,0.03065,buy,3472.859856
1,2025-09-25 00:00:00.198,113307.01,0.26476,buy,29999.163968
2,2025-09-25 00:00:00.221,113307.00,0.00142,sell,160.895940
3,2025-09-25 00:00:00.330,113307.00,0.00034,sell,38.524380
4,2025-09-25 00:00:00.371,113307.01,0.02647,buy,2999.236555


In [12]:
df

,timestamp,price,quantity,taker_side,quantity_usdt
0,2025-09-25 00:00:00.010,113307.01,0.03065,buy,3472.859856
1,2025-09-25 00:00:00.198,113307.01,0.26476,buy,29999.163968
2,2025-09-25 00:00:00.221,113307.00,0.00142,sell,160.895940
3,2025-09-25 00:00:00.330,113307.00,0.00034,sell,38.524380
4,2025-09-25 00:00:00.371,113307.01,0.02647,buy,2999.236555
...,...,...,...,...,...
4537498,2025-09-30 23:59:57.787,114048.94,0.00215,buy,245.205221
4537499,2025-09-30 23:59:58.491,114048.94,0.00059,buy,67.288875
4537500,2025-09-30 23:59:58.613,114048.94,0.00033,buy,37.636150
4537501,2025-09-30 23:59:58.734,114048.93,0.00027,sell,30.793211


In [13]:
df_grouped

,price,quantity_usdt,log_return
ts_1h,,,
2025-09-25 00:00:00,210.75,5.859700e+06,NaN
2025-09-25 01:00:00,208.25,1.096354e+07,-0.011933
2025-09-25 02:00:00,210.03,1.202238e+07,0.008511
2025-09-25 03:00:00,207.29,1.002807e+07,-0.013132
2025-09-25 04:00:00,204.34,1.376018e+07,-0.014334
...,...,...,...
2025-09-30 19:00:00,208.99,1.157975e+07,0.006721
2025-09-30 20:00:00,209.86,1.077816e+07,0.004154
2025-09-30 21:00:00,209.60,1.204966e+07,-0.001240


In [14]:
df['ts_1h'] = df['timestamp'].dt.ceil('1h')
df_grouped = df.groupby('ts_1h').agg({'price': 'last', 'quantity_usdt': 'sum'})
df_grouped['log_return'] = np.log(df_grouped['price'] / df_grouped['price'].shift(1))

In [15]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_grouped.index,
        y=df_grouped['log_return'],
        name='Log Returns',
        opacity=0.7,
        mode='lines'
    )
)

fig.add_trace(
    go.Bar(
        x=df_grouped.index,
        y=df_grouped['quantity_usdt'],
        name='Volume USD',
        opacity=0.7,
        yaxis='y2'
    )
)

# --- Layout with dual axes ---
fig.update_layout(
    title="5-Min Log Returns (left) vs Volume (right)",
    xaxis=dict(title="Time"),
    
    yaxis=dict(
        title="Log Returns",
        side="left"
    ),
    yaxis2=dict(
        title="Volume",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    
    bargap=0,
    template="plotly_dark",
    legend=dict(orientation="h")
)

fig.show(renderer='browser')

In [ ]:
df_grouped